# Graph Data Science in Python: four questions graphs answer

A companion to the Broadcom Graphathon 2026 workshop. It runs on a small, fictional **air-freight network**
so you can see each algorithm do something useful without the answer being your own project. Every section
ends with a *Translate it* question: what is the equivalent in your project?

| Section | Question | Algorithms |
|---|---|---|
| 1 | What matters most? | Degree, PageRank (weighted and personalized), Betweenness |
| 2 | What breaks if this fails? | WCC, Articulation Points, Bridges |
| 3 | How do things travel? | BFS, Dijkstra, Yen's k-shortest paths |
| 4 | What belongs together? | Louvain, Node Similarity, painting labels from seeds, FastRP + KNN |

Cells marked **(optional)** are skipped in the live demo but worth running on your own.

**Requirements:** Neo4j 5.x or 2025.x+ with the GDS plugin (2.13 or later), Python 3.10+, and the packages in
`requirements.txt`. Connection settings come from a `.env` file (copy `.env.example`).

## 0. Connect and load the data

In [31]:
import logging
import os
import pandas as pd
from dotenv import load_dotenv
from graphdatascience import GraphDataScience

# Hide server notifications (e.g. deprecation notices) so results stay readable. Errors still show.
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

load_dotenv()
gds = GraphDataScience(
    os.getenv("NEO4J_URI", "neo4j://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "")),
    database=os.getenv("NEO4J_DATABASE", "neo4j"),
    arrow=False,  # plain Bolt: works everywhere, and this graph is small
)
pd.set_option("display.max_colwidth", 120)
print("GDS server version:", gds.server_version())

GDS server version: 2026.9.0


In [32]:
from load_data import load, COUNTS

load(lambda q, p: gds.run_cypher(q, p))   # wipes and reloads only the workshop labels
gds.run_cypher(COUNTS)

,label,count
0,Airport,46
1,Carrier,6
2,Contract,144
3,Route,81
4,Shipper,16


### The stored model

```
(:Shipper)-[:HAS_CONTRACT]->(:Contract)-[:SHIPS_ON]->(:Route)-[:FROM|TO]->(:Airport)
(:Carrier)-[:OPERATES]->(:Route)
```

Routes are nodes because they carry their own data (distance, weekly flights, whether they're active) and
because contracts and carriers attach to them. That's a good model for storing and querying. It is not the
graph the algorithms want, as the next two cells show.

In [33]:
gds.run_cypher("""
MATCH (s:Shipper)-[:HAS_CONTRACT]->(c:Contract)-[:SHIPS_ON]->(r:Route)-[:FROM]->(a:Airport),
      (r)-[:TO]->(b:Airport)
RETURN s.name AS shipper, s.industry AS industry, c.annualRevenueM AS revenueM,
       a.code + ' - ' + b.code AS route, r.distanceKm AS km, r.weeklyFlights AS flights
ORDER BY revenueM DESC LIMIT 5
""")

,shipper,industry,revenueM,route,km,flights
0,Helix Pharma,Pharma,21.83,CRV - EMB,288,6
1,Helix Pharma,Pharma,21.83,CRV - DUN,956,7
2,Novacure Biologics,Pharma,21.15,CRV - ASG,338,8
3,Helix Pharma,Pharma,21.08,CRV - WSP,1806,16
4,Helix Pharma,Pharma,21.08,CRV - BWT,222,7


## Projections: reshape before you run

GDS doesn't run on the database. It runs on an in-memory **projection**, and the projection doesn't have to look
like the stored graph.

**A native projection mirrors storage.** Projecting airports and routes as they're stored gives a graph where
airports never touch each other directly: every connection goes through a Route node.

In [34]:
gds.graph.drop("stored", fail_if_missing=False)
stored, result = gds.graph.project.native(
    "stored",
    ["Airport", "Route"],
    {"FROM": {"orientation": "UNDIRECTED"}, "TO": {"orientation": "UNDIRECTED"}},
)
print(f"{result.node_count} nodes (airports + routes), {result.relationship_count} relationships")
stored.drop()

127 nodes (airports + routes), 324 relationships


GraphInfo(graph_name='stored', database='neo4j', database_location='local', configuration={'relationshipProjection': {'FROM': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'FROM', 'properties': {}, 'indexInverse': False}, 'TO': {'orientation': 'UNDIRECTED', 'aggregation': 'DEFAULT', 'type': 'TO', 'properties': {}, 'indexInverse': False}}, 'jobId': '7c64fd51-3ee0-4fe8-b6c3-192fbb4af9e8', 'nodeProjection': {'Route': {'properties': {}, 'label': 'Route'}, 'Airport': {'properties': {}, 'label': 'Airport'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, memory_usage='', size_in_bytes=-1, node_count=127, relationship_count=324, creation_time=datetime.datetime(2026, 10, 15, 13, 18, 49, 969431, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), modification_time=datetime.datetime(2026, 10, 15, 13, 18, 49, 969431, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17

**A Cypher projection reshapes on the way in.** This one collapses `Airport <- Route -> Airport` into a single
weighted `FLOW` relationship between airports, drops suspended routes, and adds up the contract revenue on
each route. One query, and the graph now answers questions about airports.

In [35]:
NETWORK_QUERY = """
MATCH (a:Airport)<-[:FROM]-(r:Route)-[:TO]->(b:Airport)
WHERE r.active AND NOT a.code IN $closed AND NOT b.code IN $closed
OPTIONAL MATCH (r)<-[:SHIPS_ON]-(c:Contract)
WITH a, b, r, sum(c.annualRevenueM) AS revenue
RETURN gds.graph.project(
  $graph_name, a, b,
  {
    sourceNodeLabels: labels(a),
    targetNodeLabels: labels(b),
    relationshipType: 'FLOW',
    relationshipProperties: {
      revenue: revenue,
      distanceKm: toFloat(r.distanceKm),
      weeklyFlights: toFloat(r.weeklyFlights)
    }
  },
  {undirectedRelationshipTypes: ['*']}
)
"""

def project_network(name="network", closed=()):
    """Project the airport network, optionally with some airports closed."""
    gds.graph.drop(name, fail_if_missing=False)
    G, result = gds.graph.project.cypher(NETWORK_QUERY, graph_name=name, closed=list(closed))
    print(f"{name}: {result.node_count} airports, {result.relationship_count} relationships (both directions)")
    return G

G = project_network()

network: 46 airports, 160 relationships (both directions)


GDS returns internal node ids. This helper adds airport codes and names to any result.

In [37]:
def airport_lookup(ids):
    return gds.run_cypher(
        "UNWIND $ids AS id WITH id, gds.util.asNode(id) AS a "
        "RETURN id AS nodeId, a.code AS code, a.name AS name, a.region AS region, a.type AS type",
        {"ids": [int(i) for i in ids]},
    ).set_index("nodeId")

AIRPORTS = airport_lookup(gds.degree.stream(G)["nodeId"])

def named(df, col="nodeId", prefix=""):
    """Insert code/name/region columns next to a node id column."""
    out = df.copy()
    for field in ["type", "region", "name", "code"]:
        out.insert(out.columns.get_loc(col) + 1, prefix + field, out[col].map(AIRPORTS[field]))
    return out.drop(columns=col)

def codes(node_ids):
    return " -> ".join(AIRPORTS.loc[int(i), "code"] for i in node_ids)

NODE_IDS = {code: node for node, code in AIRPORTS["code"].items()}

def node_id(code):
    return int(NODE_IDS[code])

---
## 1. What matters most?

Four ways to rank airports, each answering a slightly different question.

In [38]:
degree = gds.degree.stream(G).rename(columns={"score": "routes"})
flights = gds.degree.stream(G, relationship_weight_property="weeklyFlights").rename(columns={"score": "weeklyFlights"})
pagerank = gds.page_rank.stream(G).rename(columns={"score": "pageRank"})
revenue_rank = gds.page_rank.stream(G, relationship_weight_property="revenue").rename(columns={"score": "revenuePageRank"})

def top(df, col, n=5):
    return named(df.sort_values(col, ascending=False).head(n))[["code", "name", col]].reset_index(drop=True)

pd.concat(
    {"Most routes": top(degree, "routes"),
     "Most weekly flights": top(flights, "weeklyFlights"),
     "PageRank (unweighted)": top(pagerank, "pageRank"),
     "PageRank (by revenue)": top(revenue_rank, "revenuePageRank")},
    axis=1,
).round(3)

Most routes                     Most weekly flights             \
         code         name routes                code       name   
0         NGT    Northgate   18.0                 NGT  Northgate   
1         CRV      Corvina   14.0                 CRV    Corvina   
2         SOL       Solano   11.0                 SOL     Solano   
3         WSP     Westport   11.0                 WSP   Westport   
4         MRI  Marrow Isle    4.0                 MRG   Marigold   

                PageRank (unweighted)                        \
  weeklyFlights                  code         name pageRank   
0         433.0                   NGT    Northgate    4.356   
1         124.0                   CRV      Corvina    3.313   
2          94.0                   WSP     Westport    2.702   
3          91.0                   SOL       Solano    2.512   
4          41.0                   MRI  Marrow Isle    1.249   

  PageRank (by revenue)                                 
                   code           name revenuePageRank  
0                   CRV        Corvina           8.216  
1                   SOL         Solano           4.367  
2                   NGT      Northgate           3.643  
3                   WSP       Westport           2.416  
4                   SLV  Silica Valley           2.192

**How to read this.** Northgate (NGT) has the most routes and by far the most flights, and plain PageRank agrees.
Weight the routes by the contract revenue they carry and Corvina (CRV), the pharma hub, moves to the top while
Northgate drops. Silica Valley (SLV), a regional airport, appears too.

Read the **ranking**, not the numbers: PageRank scores only compare within a single run.

### Personalized PageRank: important *to whom*?

Start the random walk only from the airports that one customer (Helix Pharma) uses. The scores become
importance *relative to that customer*.

In [17]:
helix_airports = gds.run_cypher("""
MATCH (:Shipper {name: 'Helix Pharma'})-[:HAS_CONTRACT]->(:Contract)-[:SHIPS_ON]->(:Route)-[:FROM|TO]->(a:Airport)
RETURN DISTINCT a.code AS code
""")["code"].tolist()
print("Helix Pharma ships through:", ", ".join(sorted(helix_airports)))

helix_rank = gds.page_rank.stream(
    G,
    relationship_weight_property="revenue",
    source_nodes=[node_id(c) for c in helix_airports],
)
top(helix_rank, "score", 8)

Helix Pharma ships through: BWT, CDF, CRV, DLM, DUN, EMB, FHV, GRS, KES, NGT, SOL, WSP


,code,name,score
0,CRV,Corvina,3.573730
1,SOL,Solano,1.277465
2,NGT,Northgate,0.726352
3,WSP,Westport,0.713583
4,FHV,Fairhaven,0.665848
5,SLV,Silica Valley,0.586531
6,EMB,Emberly,0.551823
7,BWT,Brightwater,0.376048


### Betweenness: who sits in the middle?

Betweenness counts how many shortest paths pass through a node. Watch for nodes with **few connections but high
betweenness**: they are brokers.

In [18]:
betweenness = gds.betweenness.stream(G).merge(degree, on="nodeId")
named(betweenness.sort_values("score", ascending=False).head(8))[["code", "name", "type", "routes", "score"]].reset_index(drop=True)

,code,name,type,routes,score
0,NGT,Northgate,Hub,18.0,496.716667
1,WSP,Westport,Hub,11.0,377.400000
2,CRV,Corvina,Hub,14.0,289.016667
3,CLR,Cape Lorne,Regional,4.0,266.000000
4,SOL,Solano,Hub,11.0,167.850000
5,MRI,Marrow Isle,Regional,4.0,147.000000
6,FJH,Fjordhaven,Regional,4.0,126.000000
7,CRH,Coral Reach,Regional,3.0,69.000000


**How to read this.** Cape Lorne (CLR) is a regional airport with only a handful of routes, yet it ranks among the
hubs. Raw betweenness values grow with graph size, so compare nodes within the same graph only.

> **Translate it:** What should "important" be measured in for your project, and important to whom?

---
## 2. What breaks if this fails?

In [19]:
wcc = gds.wcc.stats(G)
print("Connected components:", wcc.component_count)
print("Component sizes:", wcc.component_distribution)

Connected components: 1
Component sizes: {'p1': 46, 'max': 46, 'p5': 46, 'p90': 46, 'p50': 46, 'p95': 46, 'p10': 46, 'p75': 46, 'p99': 46, 'p25': 46, 'min': 46, 'mean': 46.0, 'p999': 46}


**How to read this.** Run WCC first on any new graph. One component means everything can reach everything.
Several components, or unexpected isolated nodes, usually point to a data or projection problem, and they
quietly distort other algorithms.

### Articulation points: single points of failure

An articulation point is a node whose removal splits the graph. Recent GDS versions also report the resulting
pieces; the smallest piece is what gets cut off.

In [20]:
ap = gds.articulation_points.stream(G).merge(degree, on="nodeId")
if "resultingComponents" in ap.columns:   # GDS versions that report the pieces
    ap = ap[ap["resultingComponents"].notna()].copy()
    ap["airportsCutOff"] = ap["resultingComponents"].map(lambda c: c["min"])
    ap = ap.drop(columns="resultingComponents").sort_values("airportsCutOff", ascending=False)
named(ap).drop(columns=["type", "region"]).reset_index(drop=True)

,code,name,routes,airportsCutOff
0,CLR,Cape Lorne,4.0,7
1,FJH,Fjordhaven,4.0,3
2,CRV,Corvina,14.0,2
3,NGT,Northgate,18.0,1
4,SOL,Solano,11.0,1
5,WSP,Westport,11.0,1
6,TLV,Tolvik,3.0,1


**How to read this.** The list itself is the answer, so rank it by what each node would cut off. The hubs cut off a
single spoke airport or two. Cape Lorne (CLR), with only a few routes, would cut off the entire seven-airport
archipelago.

### Bridges: single points of failure on links (optional)

In [21]:
bridges = gds.bridges.stream(G)
bridges["route"] = [f"{AIRPORTS.loc[int(a), 'code']} - {AIRPORTS.loc[int(b), 'code']}" for a, b in zip(bridges["from"], bridges["to"])]
bridges[["route"] + [c for c in bridges.columns if c == "remainingSizes"]]

,route,remainingSizes
0,WSP - YRW,"[45, 1]"
1,SOL - QPT,"[45, 1]"
2,NGT - IVL,"[45, 1]"
3,TLV - ULB,"[45, 1]"


> **Translate it:** Where is the single point of failure in your graph, and what would it cut off?

---
## 3. How do things travel?

### BFS: what can I reach? (optional)

In [22]:
reach = gds.bfs.stream(G, source_node=node_id("MRI"), max_depth=2)
print("Reachable from Marrow Isle within 2 hops:", codes(reach["nodeIds"][0]).replace(" -> ", ", "))

Reachable from Marrow Isle within 2 hops: MRI, CLR, KPS, PCY, CRH, WSP, RDG, TRN, GRK


### Dijkstra: the cheapest path

"Cheap" means whatever the weight measures: here, distance. It could be cost, latency or risk. Weights must be
non-negative, so decide what high and low mean before you use one.

In [23]:
def cheapest(graph, source, target):
    df = gds.shortest_path.dijkstra.stream(graph, source_node=node_id(source), target_nodes=node_id(target),
                                           relationship_weight_property="distanceKm")
    return pd.DataFrame({"path": [codes(ids) for ids in df["nodeIds"]], "totalKm": df["totalCost"]})

cheapest(G, "MRI", "PLS")

,path,totalKm
0,MRI -> CLR -> WSP -> SOL -> PLS,2399.0


### A hub closes

Westport (WSP) is on the cheapest path. Close it by **projecting the network again without it** (projections are
cheap, so a "what if" is just another projection), then ask for the cheapest reroute and the next-best alternatives.

In [24]:
G_closed = project_network("network_wsp_closed", closed=["WSP"])

yens = gds.shortest_path.yens.stream(G_closed, source_node=node_id("MRI"), target_node=node_id("PLS"), k=3,
                                     relationship_weight_property="distanceKm")
pd.DataFrame({"option": yens["index"] + 1, "path": [codes(ids) for ids in yens["nodeIds"]], "totalKm": yens["totalCost"]})

network_wsp_closed: 44 airports, 138 relationships (both directions)


,option,path,totalKm
0,1,MRI -> CLR -> RDG -> NGT -> PLS,3898.0
1,2,MRI -> CLR -> RDG -> NGT -> SOL -> PLS,3906.0
2,3,MRI -> CLR -> RDG -> NGT -> MRG -> SOL -> PLS,4039.0


**How to read this.** `totalCost` is in the units of your weight (km here). The reroute goes through Ridgeway (RDG)
and Northgate, adding about 1,500 km. Sanity-check that a path makes sense before trusting it.

> **Translate it:** What travels through your graph, and what should make a path "expensive"?

In [ ]:
G_closed.drop()

---
## 4. What belongs together?

### Louvain: communities nobody told the algorithm about

In [25]:
louvain = gds.louvain.stats(G)
print(f"Modularity {louvain.modularity:.2f} across {louvain.community_count} communities")

communities = named(gds.louvain.stream(G))
pd.crosstab(communities["communityId"], communities["region"])

Modularity 0.57 across 6 communities


region,Archipelago,East,North,South,West
communityId,,,,,
4,0,0,8,0,0
6,0,0,4,0,0
15,0,9,0,0,0
21,0,0,0,9,0
31,0,0,0,0,7
38,7,0,0,0,2


**How to read this.** Run `stats` first: modularity above roughly 0.3 suggests real community structure. Louvain was
never told the regions, yet its communities line up with them, and Cape Lorne typically lands with the archipelago it serves.
Check the community sizes too; one giant community usually means the projection is too dense.

### Node Similarity: who could stand in for whom?

Node Similarity compares neighbor sets (Jaccard). Two airports with the same connections are natural substitutes.
`degree_cutoff` skips airports with too few routes to compare meaningfully.

In [26]:
similar = gds.node_similarity.stream(G, top_k=1, similarity_cutoff=0.5, degree_cutoff=2)
similar = named(named(similar, "node1", "a_"), "node2", "b_")
similar["pair"] = [tuple(sorted(p)) for p in zip(similar["a_code"], similar["b_code"])]
(similar.drop_duplicates("pair").sort_values("similarity", ascending=False)
        [["a_code", "a_name", "b_code", "b_name", "similarity"]].head(6).reset_index(drop=True))

,a_code,a_name,b_code,b_name,similarity
0,DUN,Dunmore,KES,Kessa,1.000000
1,SLV,Silica Valley,DUN,Dunmore,0.750000
2,STW,Stillwater,WLM,Willowmere,0.666667
3,MRG,Marigold,NVR,Novara,0.600000
4,FJH,Fjordhaven,HRF,Harrowfield,0.500000
5,ASG,Ashgrove,DLM,Delmont,0.500000


**How to read this.** Scores run from 0 to 1. Dunmore (DUN) and Kessa (KES) score 1.0: they connect to exactly the
same airports. Pick a cutoff, then eyeball a few pairs to check that "similar" means what you hoped.

### Painting labels from a few seeds

Suppose we only know one airport per trade corridor: Corvina for pharma, Silica Valley for semiconductors, Marrow Isle
for perishables. Which corridor does every other airport belong to?

This reuses Personalized PageRank from section 1: run it once per seed over revenue-weighted routes, convert each run to
shares (so runs are comparable), and give each airport the corridor it is most tied to. Airports with no clear winner
are marked *Mixed*.

In [27]:
SEEDS = {"Pharma": "CRV", "Semiconductors": "SLV", "Perishables": "MRI"}

shares = {}
for corridor, seed in SEEDS.items():
    run = gds.page_rank.stream(G, relationship_weight_property="revenue", source_nodes=[node_id(seed)])
    shares[corridor] = run.set_index("nodeId")["score"] / run["score"].sum()

painted = pd.DataFrame(shares)
painted.index.name = "nodeId"
painted["confidence"] = painted.max(axis=1) / painted.sum(axis=1)
painted["corridor"] = painted[list(SEEDS)].idxmax(axis=1).where(painted["confidence"] >= 0.45, "Mixed")
painted = named(painted.reset_index())
pd.crosstab(painted["region"], painted["corridor"])

corridor,Mixed,Perishables,Pharma,Semiconductors
region,,,,
Archipelago,0,7,0,0
East,0,0,9,0
North,12,0,0,0
South,0,0,3,6
West,7,2,0,0


**How to read this.** The archipelago and Cape Lorne paint as perishables, the East as pharma, and the South mostly
as semiconductors, even though nothing in the data labels them. The North and most of the West come out *Mixed*:
they're tied to several corridors at once, which is an answer too. The *confidence* column is what separates a
painted label from a guess, so look at the low-confidence airports by hand.

GDS also has **Label Propagation**, which can be seeded the same way. It is very fast, but on a graph this connected
one label can flood the rest, and results can change between runs. If you use it, run it twice and compare.

### FastRP + KNN: neighborhood look-alikes

FastRP turns each airport's network neighborhood into a vector. KNN then finds the nearest vectors. Here we chain
algorithms in memory with `mutate`, then `write` the result back to the database.

In [28]:
if "embedding" not in G.node_properties().get("Airport", []):
    gds.fast_rp.mutate(G, mutate_property="embedding", embedding_dimension=32, random_seed=42)

knn = gds.knn.stream(G, node_properties=["embedding"], top_k=3, random_seed=42, concurrency=1)
knn = named(named(knn, "node1", "a_"), "node2", "b_")
knn[knn["a_code"].isin(["CLR", "DUN", "NGT"])][["a_code", "b_code", "b_name", "similarity"]].round(3)

,a_code,b_code,b_name,similarity
0,NGT,IVL,Ivel,0.974
1,NGT,BRK,Brennick,0.953
2,NGT,HRF,Harrowfield,0.950
60,DUN,KES,Kessa,1.000
61,DUN,SLV,Silica Valley,0.996
62,DUN,NVR,Novara,0.992
84,CLR,RDG,Ridgeway,0.946
85,CLR,PCY,Pearl Cay,0.940
86,CLR,MRI,Marrow Isle,0.909


**How to read this.** Similarity here means "has a similar network neighborhood", which mostly means nearby and
similarly connected. Like any similarity score, it needs a cutoff and a sanity check.

> **Translate it:** What do you know about a few nodes that you wish you knew about all of them?

---
## 5. Write what you need back to the database

`stream` is for looking; `write` is for keeping. Once results are properties and relationships in Neo4j, any Cypher
query, dashboard or AI agent can use them.

In [29]:
gds.page_rank.write(G, write_property="revenueRank", relationship_weight_property="revenue")
gds.louvain.write(G, write_property="community")
gds.knn.write(G, write_relationship_type="SIMILAR", write_property="score",
              node_properties=["embedding"], top_k=3, random_seed=42, concurrency=1)

gds.run_cypher("""
MATCH (a:Airport)
WITH a.community AS community, a ORDER BY a.revenueRank DESC
WITH community, collect(a.code)[..3] AS topAirports, count(*) AS airports
RETURN community, airports, topAirports ORDER BY airports DESC
""")

,community,airports,topAirports
0,15,9,"[CRV, FHV, EMB]"
1,21,9,"[SOL, SLV, DUN]"
2,38,9,"[CLR, MRI, CRH]"
3,4,8,"[NGT, DLN, GLM]"
4,31,7,"[WSP, THB, STW]"
5,6,4,"[FJH, TLV, ULB]"


## 6. Clean up

Projections live in heap memory until dropped. Note also that a projection is a snapshot: properties written above
won't appear in `G` unless you project again.

In [30]:
G.drop()
gds.graph.list()

[GraphInfoWithDegrees(graph_name='network_wsp_closed', database='neo4j', database_location='local', configuration={'jobId': 'jid-1b25e0e4-3f1d-455a-8443-cad9b7518e84', 'inverseIndexedRelationshipTypes': [], 'logProgress': True, 'undirectedRelationshipTypes': ['*'], 'readConcurrency': 4, 'query': "\nMATCH (a:Airport)<-[:FROM]-(r:Route)-[:TO]->(b:Airport)\nWHERE r.active AND NOT a.code IN $closed AND NOT b.code IN $closed\nOPTIONAL MATCH (r)<-[:SHIPS_ON]-(c:Contract)\nWITH a, b, r, sum(c.annualRevenueM) AS revenue\nRETURN gds.graph.project(\n  $graph_name, a, b,\n  {\n    sourceNodeLabels: labels(a),\n    targetNodeLabels: labels(b),\n    relationshipType: 'FLOW',\n    relationshipProperties: {\n      revenue: revenue,\n      distanceKm: toFloat(r.distanceKm),\n      weeklyFlights: toFloat(r.weeklyFlights)\n    }\n  },\n  {undirectedRelationshipTypes: ['*']}\n)\n"}, memory_usage='16 KiB', size_in_bytes=16432, node_count=44, relationship_count=138, creation_time=datetime.datetime(2026, 10

---
## Try it with an agent

Install the Neo4j agent skills into your coding agent (see the README), point it at this database and ask open
questions. Some to start with:

- *Which airports are single points of failure, and what would each one cut off?*
- *Rank airports by the contract revenue that depends on them. How different is that from ranking by flights?*
- *If Corvina closed for a week, which shippers would be affected and what are the cheapest reroutes?*

Watch what it does: whether it checks the GDS version, which algorithm it picks, and whether it streams before it writes.